# CareerPilot AI — Resume Parsing Pipeline

Run every cell top to bottom, in order. Each step prints its own output so you can see exactly what happened before moving to the next one.

**What this notebook does:** upload a resume -> extract its text -> parse name/skills with spaCy -> save the structured profile to SQLite so the rest of the team can use it.

## Step 0 — Install and import everything

In [2]:
!pip install pdfplumber python-docx spacy ipywidgets --quiet
!python -m spacy download en_core_web_sm --quiet

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [3]:
import os
import json
import sqlite3
import pdfplumber
from docx import Document
import spacy
from spacy.matcher import PhraseMatcher
import ipywidgets as widgets
from IPython.display import display

UPLOAD_DIR = "uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)

print("Setup complete.")

Setup complete.


## Step 1 — Upload your resume

Run the cell below. A button will appear — click **Upload** and choose your resume (PDF or DOCX).

If the button doesn't appear or doesn't work in your notebook environment, skip to the **manual fallback** in the cell after — just drag your resume file into the file browser panel on the left side of the notebook, then set the path by hand.

In [4]:
uploader = widgets.FileUpload(accept=".pdf,.docx", multiple=False)
display(uploader)

FileUpload(value=(), accept='.pdf,.docx', description='Upload')

In [6]:
# Run this AFTER selecting a file in the cell above.
file_path = None

if len(uploader.value) > 0:
    uploaded = uploader.value
    if isinstance(uploaded, tuple):          # ipywidgets 8+
        item = uploaded[0]
        name = item["name"]
        content = item["content"]
    else:                                     # ipywidgets 7.x
        name = list(uploaded.keys())[0]
        content = uploaded[name]["content"]

    file_path = os.path.join(UPLOAD_DIR, name)
    with open(file_path, "wb") as f:
        f.write(bytes(content))
    print(f"Saved: {file_path}")
else:
    print("No file uploaded yet.")
    print("uploader.value was:", uploader.value)   # debug line — shows exactly what the widget saw

No file uploaded yet.
uploader.value was: ()


In [5]:
import time
start = time.time()

raw_text = extract_text(file_path)

print(f"Took {time.time() - start:.2f} seconds")
print(f"Extracted {len(raw_text)} characters")
print(raw_text[:300])

NameError: name 'extract_text' is not defined

## Step 2 — Extract raw text from the file

PDFs and DOCX files store text differently, so we need a separate reader for each.

In [14]:
# file_path = "uploads/Hemanth_Resume_110726.pdf"   # match the exact name from your listdir() output
# print("file_path is now:", file_path)

In [ ]:
def extract_text(path: str) -> str:
    if path.endswith(".pdf"):
        text = ""
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                text += page.extract_text() or ""
        return text
    elif path.endswith(".docx"):
        doc = Document(path)
        return "\n".join(p.text for p in doc.paragraphs)
    else:
        raise ValueError("Unsupported file type — use PDF or DOCX")

raw_text = extract_text(file_path)
print(raw_text[:500])

## Step 3 — Parse name, organizations, and locations with spaCy

spaCy's Named Entity Recognition (NER) scans the text and tags spans it recognizes as people, organizations, or places.

In [8]:
nlp = spacy.load("en_core_web_sm")

def get_entities(text: str) -> dict:
    doc = nlp(text)
    name = None
    organizations = []
    locations = []
    for ent in doc.ents:
        if ent.label_ == "PERSON" and name is None:
            name = ent.text
        elif ent.label_ == "ORG":
            organizations.append(ent.text)
        elif ent.label_ == "GPE":
            locations.append(ent.text)
    return {"name": name, "organizations": organizations, "locations": locations}

entities = get_entities(raw_text)
entities

{'name': 'MUMMAREDDY HEMANTH',
 'organizations': ['Java & DSA',
  'Computer Science Undergraduate',
  'AI & ML',
  'Computer Science',
  'AI & ML',
  'Object-Oriented Programming',
  'Data Structures &\nAlgorithms',
  'Smart India\n',
  'Software Development Engineer',
  'Object-Oriented Programming',
  'Data Structures & Algorithms',
  'SQL',
  'CSS',
  'JavaScript',
  'React.js\nTools & Version Control: Git',
  'JWT',
  'Data Structures & Algorithms',
  'SmartBridge',
  'PROJECTS\nFarmxchain — Agriculture',
  'JWT',
  'HTML',
  'CSS',
  'JavaScript',
  'JWT',
  'Library Management System Mar',
  'AI-Powered Resume Screener',
  'ML microservice',
  'Python ML',
  'API',
  'CSS',
  'JavaScript',
  'CERTIFICATIONS',
  'Python- NPTEL',
  'SVCE',
  'Computer Science (AI & ML',
  'Sri Venkateswara College of Engineering Jun 2023',
  'MPC',
  'Sreenivasa Junior College'],
 'locations': ['Tirupati',
  'India',
  'linkedin.com/in/hemanthkumarreddy',
  'Farmxchain',
  'AI',
  'AI',
  'Spring',

## Step 4 — Match known skills

This is a plain keyword list — expand it with more skills relevant to your test resumes as you go. Matching is case-insensitive.

In [9]:
SKILLS = [
    "Python", "Java", "JavaScript", "SQL", "React", "FastAPI", "Flask",
    "Machine Learning", "Deep Learning", "AWS", "Docker", "Kubernetes",
    "Git", "spaCy", "MongoDB", "PostgreSQL", "TensorFlow", "PyTorch",
    "Node.js", "HTML", "CSS", "C++", "Data Analysis", "NLP",
]

matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
matcher.add("SKILLS", [nlp.make_doc(skill) for skill in SKILLS])

def get_skills(text: str) -> list:
    doc = nlp(text)
    matches = matcher(doc)
    found = {doc[start:end].text for _, start, end in matches}
    return sorted(found)

skills = get_skills(raw_text)
skills

['CSS',
 'Git',
 'HTML',
 'Java',
 'JavaScript',
 'Python',
 'SQL',
 'javascript',
 'sql']

## Step 5 — Combine everything into one structured profile

In [10]:
profile = {
    "filename": os.path.basename(file_path),
    "name": entities["name"],
    "organizations": entities["organizations"],
    "locations": entities["locations"],
    "skills": skills,
    "raw_text": raw_text,
}

print(json.dumps(profile, indent=2))

{
  "filename": "Hemanth_Resume_110726.pdf",
  "name": "MUMMAREDDY HEMANTH",
  "organizations": [
    "Java & DSA",
    "Computer Science Undergraduate",
    "AI & ML",
    "Computer Science",
    "AI & ML",
    "Object-Oriented Programming",
    "Data Structures &\nAlgorithms",
    "Smart India\n",
    "Software Development Engineer",
    "Object-Oriented Programming",
    "Data Structures & Algorithms",
    "SQL",
    "CSS",
    "JavaScript",
    "React.js\nTools & Version Control: Git",
    "JWT",
    "Data Structures & Algorithms",
    "SmartBridge",
    "PROJECTS\nFarmxchain \u2014 Agriculture",
    "JWT",
    "HTML",
    "CSS",
    "JavaScript",
    "JWT",
    "Library Management System Mar",
    "AI-Powered Resume Screener",
    "ML microservice",
    "Python ML",
    "API",
    "CSS",
    "JavaScript",
    "CERTIFICATIONS",
    "Python- NPTEL",
    "SVCE",
    "Computer Science (AI & ML",
    "Sri Venkateswara College of Engineering Jun 2023",
    "MPC",
    "Sreenivasa Junior 

## Step 6 — Save the profile to SQLite

This is the file the rest of your team (schema + matching) will read from.

In [11]:
DB_PATH = "careerpilot.db"

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute('''
    CREATE TABLE IF NOT EXISTS career_profiles (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        filename TEXT,
        name TEXT,
        organizations TEXT,
        locations TEXT,
        skills TEXT,
        raw_text TEXT
    )
''')

cur.execute('''
    INSERT INTO career_profiles (filename, name, organizations, locations, skills, raw_text)
    VALUES (?, ?, ?, ?, ?, ?)
''', (
    profile["filename"],
    profile["name"],
    json.dumps(profile["organizations"]),
    json.dumps(profile["locations"]),
    json.dumps(profile["skills"]),
    profile["raw_text"],
))

conn.commit()
print(f"Saved profile to {DB_PATH}")

Saved profile to careerpilot.db


## Step 7 — Verify it saved correctly

In [12]:
cur.execute("SELECT id, filename, name, skills FROM career_profiles ORDER BY id DESC LIMIT 1")
row = cur.fetchone()

print("Latest saved profile:")
print(f"  ID: {row[0]}")
print(f"  File: {row[1]}")
print(f"  Name: {row[2]}")
print(f"  Skills: {row[3]}")

conn.close()

Latest saved profile:
  ID: 1
  File: Hemanth_Resume_110726.pdf
  Name: MUMMAREDDY HEMANTH
  Skills: ["CSS", "Git", "HTML", "Java", "JavaScript", "Python", "SQL", "javascript", "sql"]
